# 05 - Hyperparameter Tuning (Random Forest)

Step 5 tunes the best shortlist model (Random Forest) using RandomizedSearchCV with PR-AUC.


In [ ]:
import os
import pickle
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import average_precision_score, precision_recall_curve

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (10, 6)

In [ ]:
# ---------- Global Paths ----------
# Reproducible root: run notebook from project root directory.
PROJECT_ROOT = os.getcwd()

STEP3_TABLE_DIR = os.path.join(PROJECT_ROOT, "step3_prepare", "tables")
STEP4_MODEL_DIR = os.path.join(PROJECT_ROOT, "step4_modeling", "models")

STEP5_DIR = os.path.join(PROJECT_ROOT, "step5_finetune")
STEP5_PLOT_DIR = os.path.join(STEP5_DIR, "plots")
STEP5_TABLE_DIR = os.path.join(STEP5_DIR, "tables")
STEP5_MODEL_DIR = os.path.join(STEP5_DIR, "models")

os.makedirs(STEP5_PLOT_DIR, exist_ok=True)
os.makedirs(STEP5_TABLE_DIR, exist_ok=True)
os.makedirs(STEP5_MODEL_DIR, exist_ok=True)

X_TRAIN_PATH = os.path.join(STEP3_TABLE_DIR, "X_train_processed.pkl")
X_TEST_PATH = os.path.join(STEP3_TABLE_DIR, "X_test_processed.pkl")
Y_TRAIN_PATH = os.path.join(STEP3_TABLE_DIR, "y_train.pkl")
Y_TEST_PATH = os.path.join(STEP3_TABLE_DIR, "y_test.pkl")

BASELINE_RF_PATH = os.path.join(STEP4_MODEL_DIR, "random_forest_model.pkl")

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Step5 plot dir:", STEP5_PLOT_DIR)
print("Step5 table dir:", STEP5_TABLE_DIR)
print("Step5 model dir:", STEP5_MODEL_DIR)

In [ ]:
# ---------- Helper Functions ----------
def load_pickle(path: str):
    with open(path, "rb") as f:
        return pickle.load(f)


def save_pickle(obj, path: str) -> None:
    with open(path, "wb") as f:
        pickle.dump(obj, f)
    print(f"Saved model: {path}")


def save_table(df: pd.DataFrame, filename: str) -> str:
    out = os.path.join(STEP5_TABLE_DIR, filename)
    df.to_csv(out, index=False)
    print(f"Saved table: {out}")
    return out


def save_plot(filename: str, dpi: int = 300) -> str:
    out = os.path.join(STEP5_PLOT_DIR, filename)
    plt.savefig(out, dpi=dpi, bbox_inches="tight")
    print(f"Saved plot: {out}")
    return out


def ensure_dataframe(X):
    return X if isinstance(X, pd.DataFrame) else pd.DataFrame(X)


def ensure_target_1d(y):
    if isinstance(y, pd.DataFrame):
        y = np.ravel(y.values)
    elif isinstance(y, pd.Series):
        y = y.values
    else:
        y = np.ravel(np.array(y))
    return y.astype(int)

## 1) Load Data and Baseline Model


In [ ]:
X_train = ensure_dataframe(load_pickle(X_TRAIN_PATH))
X_test = ensure_dataframe(load_pickle(X_TEST_PATH))
y_train = ensure_target_1d(load_pickle(Y_TRAIN_PATH))
y_test = ensure_target_1d(load_pickle(Y_TEST_PATH))

baseline_rf = load_pickle(BASELINE_RF_PATH)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train positive rate:", y_train.mean().round(4))
print("y_test positive rate:", y_test.mean().round(4))

## 2) RandomizedSearchCV Setup (5-Fold, PR-AUC)


In [ ]:
# Parameter space requested
param_distributions = {
    "n_estimators": [100, 200, 300],
    "max_depth": [10, 20, 30, None],
    "min_samples_split": [2, 5, 10],
    "max_features": ["sqrt", "log2"],
}

# Randomized search configuration
# Total grid size = 3*4*3*2 = 72. We sample a broad subset for efficiency.
n_iter_search = 30
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

rf_base_for_tuning = RandomForestClassifier(random_state=42, n_jobs=-1)

random_search = RandomizedSearchCV(
    estimator=rf_base_for_tuning,
    param_distributions=param_distributions,
    n_iter=n_iter_search,
    scoring="average_precision",
    cv=cv,
    random_state=42,
    n_jobs=-1,
    verbose=1,
    refit=True,
)

## 3) Fit Tuning Search and Retrieve Best Estimator


In [ ]:
random_search.fit(X_train, y_train)

best_rf = random_search.best_estimator_
best_params = random_search.best_params_
best_cv_ap = random_search.best_score_

print("Best parameters:", best_params)
print(f"Best CV PR-AUC (AP): {best_cv_ap:.6f}")

cv_results_df = pd.DataFrame(random_search.cv_results_)
keep_cols = [
    "params",
    "mean_test_score",
    "std_test_score",
    "rank_test_score",
]
cv_results_slim = cv_results_df[keep_cols].sort_values("rank_test_score").reset_index(drop=True)

save_table(cv_results_slim, "50_rf_randomized_search_results.csv")

## 4) Baseline vs Tuned Comparison on Test Set


In [ ]:
# Test-set AP for baseline model
y_prob_baseline = baseline_rf.predict_proba(X_test)[:, 1]
ap_baseline = average_precision_score(y_test, y_prob_baseline)

# Test-set AP for tuned model
y_prob_tuned = best_rf.predict_proba(X_test)[:, 1]
ap_tuned = average_precision_score(y_test, y_prob_tuned)

comparison_df = pd.DataFrame([
    {"model": "Baseline Random Forest", "test_ap": ap_baseline},
    {"model": "Tuned Random Forest", "test_ap": ap_tuned},
])

display(comparison_df)
save_table(comparison_df, "51_baseline_vs_tuned_test_ap.csv")

print(f"Final Test AP score (Tuned RF): {ap_tuned:.6f}")

## 5) Final PR Curve - Baseline vs Tuned


In [ ]:
precision_b, recall_b, _ = precision_recall_curve(y_test, y_prob_baseline)
precision_t, recall_t, _ = precision_recall_curve(y_test, y_prob_tuned)

baseline_rate = y_test.mean()

plt.figure(figsize=(9, 7))
plt.plot(recall_b, precision_b, linewidth=2, label=f"Baseline RF (AP={ap_baseline:.3f})")
plt.plot(recall_t, precision_t, linewidth=2, label=f"Tuned RF (AP={ap_tuned:.3f})")
plt.hlines(baseline_rate, xmin=0, xmax=1, colors="gray", linestyles="--", label=f"Baseline={baseline_rate:.3f}")

plt.title("Precision-Recall Curve: Baseline vs Tuned Random Forest")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.legend(loc="best")
plt.tight_layout()

save_plot("52_pr_curve_baseline_vs_tuned_rf.png")
plt.show()

## 6) Save Best Model Artifact


In [ ]:
best_model_path = os.path.join(STEP5_MODEL_DIR, "best_rf_model.pkl")
save_pickle(best_rf, best_model_path)

print("Best model saved to:", best_model_path)
print(f"Final Test AP score: {ap_tuned:.6f}")